# Assignment 2 - Search-enabled Gemini agent

This notebook demonstrates an agent loop: Gemini decides when to call a local document-search tool, reads the returned evidence, and then answers. The API key is loaded only from Colab Secrets.


In [ ]:
import os
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
print('API key loaded from Colab Secrets (value hidden).')


In [ ]:
!pip -q install -U google-genai
from google import genai
from google.genai import types
print('google-genai installed.')


In [ ]:
DOCUMENTS = {
    'attention_is_all_you_need': 'The Transformer uses self-attention instead of recurrence. Multi-head attention lets the model attend to different representation subspaces. Positional encodings provide token order information. The encoder-decoder architecture uses stacked attention and feed-forward layers.',
    'resnet': 'Residual networks learn a residual function with shortcut connections. The shortcut makes optimization easier in very deep networks and helps gradients flow through the model. ResNet was evaluated on image recognition benchmarks.',
    'retrieval_augmented_generation': 'Retrieval-augmented generation retrieves relevant external passages before generation. Grounding answers in retrieved evidence can reduce unsupported claims. The retriever and generator are separate components.',
    'responsible_ai': 'A trustworthy AI system should communicate uncertainty and distinguish evidence from assumptions. Evaluation should test accuracy, robustness, and harmful failure modes. Human review remains important for high-stakes use.',
}
assert len(DOCUMENTS) >= 4
print(f'{len(DOCUMENTS)} documents loaded.')


In [ ]:
def search_documents(query: str) -> str:
    """Return the two most relevant documents using simple keyword matching."""
    terms = {word.lower().strip('.,!?') for word in query.split() if len(word) > 2}
    ranked = []
    for name, text in DOCUMENTS.items():
        score = sum(term in text.lower() for term in terms)
        if score:
            ranked.append((score, name, text))
    ranked.sort(key=lambda item: item[0], reverse=True)
    if not ranked:
        return 'No relevant documents found.'
    return '\n\n'.join(f'[Source: {name}]\n{text}' for _, name, text in ranked[:2])

print(search_documents('How does attention help a Transformer?'))


In [ ]:
client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
search_tool = types.Tool(function_declarations=[types.FunctionDeclaration(
    name='search_documents',
    description='Search the supplied documents for evidence relevant to a question.',
    parameters={'type': 'OBJECT', 'properties': {'query': {'type': 'STRING'}}, 'required': ['query']}
)])
config = types.GenerateContentConfig(
    tools=[search_tool],
    system_instruction='You are a research assistant. Use search_documents when the answer depends on the supplied documents. Answer only from retrieved evidence; if evidence is missing, say so.'
)
print('Gemini agent configured.')


In [ ]:
def run_agent(question: str, max_steps: int = 6) -> str:
    contents = [types.Content(role='user', parts=[types.Part(text=question)])]
    for step in range(1, max_steps + 1):
        response = client.models.generate_content(model='gemini-2.5-flash', contents=contents, config=config)
        model_content = response.candidates[0].content
        contents.append(model_content)
        calls = [part.function_call for part in model_content.parts if part.function_call]
        if not calls:
            print(f'Step {step}: final answer')
            return response.text
        result_parts = []
        for call in calls:
            args = dict(call.args)
            result = search_documents(**args)
            print(f'Step {step}: tool call {call.name}({args["query"]!r})')
            result_parts.append(types.Part.from_function_response(name=call.name, response={'result': result}))
        contents.append(types.Content(role='user', parts=result_parts))
    return 'Stopped after reaching the maximum number of agent steps.'

print(run_agent('Why are shortcut connections useful in very deep neural networks?'))


In [ ]:
questions = [
    'How does self-attention differ from recurrence in the Transformer?',
    'What problem does retrieval-augmented generation try to reduce?',
    'What are two evaluation concerns for a trustworthy AI system?',
    'Which document mentions shortcut connections, and what do they help with?'
]
assert len(questions) >= 4
for question in questions:
    print('\n' + '=' * 70)
    print('Question:', question)
    print(run_agent(question))


## Submission checklist
- Run all cells in order in Google Colab.
- Add `GOOGLE_API_KEY` to Colab Secrets only; never paste it into a cell or commit it.
- Confirm the output shows tool calls and final answers.
- Save this notebook to a public GitHub repository and submit the repository URL.
